In [ ]:
!pip install -q pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 22.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px # For scatter map
import requests # To create http requests for geojson
import plotly.graph_objects as go # For choropleth map

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression

# Time series tools
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.ar_model import AutoReg

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pytorch_lightning as L
from sklearn.metrics import mean_absolute_error

np.random.seed(42)
torch.manual_seed(42)
L.seed_everything(42)

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
%matplotlib inline

from google.colab import drive
drive.mount('/content/gdrive')

print('All imports successful!')

INFO:lightning_fabric.utilities.seed:Seed set to 42


ValueError: mount failed

In [ ]:
def train_test_split(X, y, TRAIN_FRAC=0.8):
  """Simple test/train function that takes in
    X:
    y:
    TRAIN_FRAC:

    and outputs the training and testing data
  """
  split = int(TRAIN_FRAC * len(X))
  X_train, X_test = X[:split], X[split:]
  y_train, y_test = y[:split], y[split:]

  return X_train, X_test, y_train, y_test

In [ ]:
def compute_metrics(y_true, y_pred, label='Model'):
    """
    Compute and print RMSE and MAE for a set of predictions.

    Parameters
    ----------
    y_true : array-like, actual values
    y_pred : array-like, predicted values
    label  : str, model name for display

    Returns
    -------
    dict with keys 'RMSE' and 'MAE'
    """
    # FILL IN: compute RMSE (hint: np.sqrt + mean_squared_error)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    # FILL IN: compute MAE
    mae  = mean_absolute_error(y_true, y_pred)
    print(f'{label:<30}  RMSE={rmse:.5f}   MAE={mae:.5f}')
    return {'RMSE': rmse, 'MAE': mae}

In [ ]:
class MLPForecaster(nn.Module):
    """
    Simple Multilayer Perceptron for time series forecasting.

    Takes a lag feature vector of length `input_size` and produces
    a single scalar forecast.

    Parameters
    ----------
    input_size  : int, number of lag features
    hidden_size : int, number of neurons in each hidden layer
    """
    def __init__(self, input_size, hidden_size=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 100),       # FILL IN (MATCH)
            nn.ReLU(), # NONLINEARLY TRANSFORM
            nn.Linear(100, hidden_size // 2), # FILL IN (MATCH)
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1)  # FILL IN: output 1 value
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

In [ ]:
class RNNForecaster(nn.Module):
    """
    Single-layer Elman RNN for time series forecasting.

    Processes the lag sequence step-by-step and produces a single
    scalar forecast from the final hidden state.

    Parameters
    ----------
    input_size  : int, features per time step (1 for univariate)
    hidden_size : int, hidden state dimension
    """
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()
        # FILL IN: create an nn.RNN layer with batch_first=True
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        # FILL IN: output layer mapping hidden_size → 1
        self.fc  = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (batch, n_lags) → add feature dimension
        x = x.unsqueeze(-1)           # (batch, n_lags, 1)
        # FILL IN: pass through RNN, extract final hidden state h_n
        _, h_n = self.rnn(x)
        h_n = h_n.squeeze(0)          # (batch, hidden)
        return self.fc(h_n).squeeze(-1)

In [ ]:
class LSTMForecaster(nn.Module):
    """
    Single-layer LSTM for time series forecasting.

    Extends the RNN with a cell state and gating mechanisms, allowing
    the model to selectively retain or forget information across lags.

    Parameters
    ----------
    input_size  : int, features per time step (1 for univariate)
    hidden_size : int, hidden/cell state dimension
    """
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()
        # FILL IN: create an nn.LSTM layer with batch_first=True
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        # FILL IN: output layer mapping hidden_size → 1
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = x.unsqueeze(-1)               # (batch, n_lags, 1)
        # FILL IN: pass through LSTM, extract h_n from the returned tuple
        _, (h_n, _) = self.lstm(x)
        h_n = h_n.squeeze(0)              # (batch, hidden)
        return self.fc(h_n).squeeze(-1)

In [ ]:
def train_model(model, X_t, y_t, epochs=300, lr=1e-3, label='Model'):
    """
    Train a PyTorch model with MSE loss and Adam optimizer.

    Parameters
    ----------
    model  : nn.Module
    X_t    : torch.Tensor, (n_samples, n_lags)
    y_t    : torch.Tensor, (n_samples,)
    epochs : int
    lr     : float, learning rate
    label  : str, for display

    Returns
    -------
    list of training losses
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    losses    = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        # FILL IN: forward pass and loss
        pred = model(X_t)
        loss = loss_fn(pred, y_t)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        losses.append(loss.item())

    print(f'{label} — final loss: {losses[-1]:.6f}')
    return losses


In [ ]:
def simulate_ar1(phi, n=200, sigma=1.0, x0=0.0):
    """Simulate an AR(1) process: x[t] = phi * x[t-1] + noise."""
    x = np.zeros(n)
    x[0] = x0
    for t in range(1, n):
        x[t] = phi * x[t-1] + np.random.normal(0, sigma)
    return x

In [ ]:
def simulate_ar2(phi1, phi2, n=500, sigma=1.0):
    """Simulate an AR(2) process: x[t] = phi1*x[t-1] + phi2*x[t-2] + noise."""
    x = np.zeros(n)
    for t in range(2, n):
        x[t] = phi1 * x[t-1] + phi2 * x[t-2] + np.random.normal(0, sigma)
    return x

In [ ]:
def make_lag_matrix(series, n_lags):
    """
    Convert a 1-D time series into a lag-embedded feature matrix.

    Parameters
    ----------
    series  : array-like, shape (T,)
    n_lags  : int, number of lag features to create

    Returns
    -------
    X : np.ndarray, shape (T - n_lags, n_lags)   <- feature matrix
    y : np.ndarray, shape (T - n_lags,)           <- target vector
    """
    series = np.array(series)
    T = len(series)
    X, y = [], []

    for t in range(n_lags, T):
        # FILL IN: features are the previous n_lags values
        X.append(series[t-n_lags:t])
        # FILL IN: target is the value at time t
        y.append(series[t])

    return np.array(X), np.array(y)

In [ ]:
def make_lag_matrix_multi(df, n_lags, target_col):
    """
    Build a lag-embedded feature matrix from a multivariate time series.

    Parameters
    ----------
    df         : pd.DataFrame, shape (T, n_variables)
    n_lags     : int, number of lags per variable
    target_col : str, column name of the variable to forecast

    Returns
    -------
    X : pd.DataFrame of lag features
    y : pd.Series of targets
    """
    lagged_frames = []

    for lag in range(1, n_lags + 1):
        # FILL IN: shift the entire DataFrame by `lag` steps
        shifted = df.shift(lag)
        # FILL IN: rename columns to indicate the lag, e.g. 'passengers_log_diff_lag1'
        shifted.columns = [f'{col}_lag{lag}' for col in df.columns]
        lagged_frames.append(shifted)

    # Combine all lagged frames side by side
    feature_df = pd.concat(lagged_frames, axis=1)

    # FILL IN: the target is the current (unshifted) target column
    target = df[target_col]

    # Drop rows where any lag is NaN (the first n_lags rows)
    combined = pd.concat([feature_df, target], axis=1).dropna()
    X = combined.drop(columns=[target_col])
    y = combined[target_col]

    return X, y


In [ ]:
def ts_diagnostics(series, label='Series', lags=40, rolling_window=12):
    """
    Full diagnostic panel for a time series:
      1. Raw plot with rolling mean and std
      2. ADF test result shown in axis label
      3. ACF and PACF plots
    """

    series = series.dropna() # Drop NAN values in the data
    fig, axes = plt.subplots(3, 1, figsize = (12, 10))

    # Calculate rolling mean and std
    roll_mean = series.rolling(rolling_window).mean()
    roll_std = series.rolling(rolling_window).std()

    # Plot
    axes[0].plot(series, label = "Series")
    axes[0].plot(roll_mean, label = f"{rolling_window}-period Mean", color = "crimson")
    axes[0].plot(roll_std, label = f"{rolling_window}-period Std", color = "orange")
    axes[0].set_xlabel('Value')
    axes[0].set_title(f"{label}: Raw + Rolling Stats")

    adf_result = adfuller(series)
    p = adf_result[1]
    status = "STATIONARY" if p < 0.05 else "NON-STATIONARY"
    axes[0].set_xlabel(f"ADF p-value = {p:.4f} -> {status}")

    # Plot 2 ACF
    plot_acf(series, lags = lags, ax = axes[1])
    axes[1].set_title(f"{label}: ACF")

    # Plot 3 PACF
    plot_pacf(series, lags = lags, ax = axes[2], method = 'ywm')
    axes[2].set_title(f"{label}: PACF")

    plt.tight_layout()
    plt.show()

In [ ]:
def run_adf(series, label='Series'):
    """Run the ADF test and print a readable summary."""
    result = adfuller(series.dropna())
    print(f'The ADF test: {label}')
    print(f'ADF Statistics: {result[0]:.4f}')
    print(f'p value: {result[1]: .4f}')
    for key, val in result[4].items():
      print(f'{key}: {val: .4f}')
    if(result[1] < 0.05):
      print(' --> STATIONARY (REJECT H0)')
    else:
      print('--> NON STAIONARY (FAIL TO REJECT)')
    ...



In [ ]:
def RMSE(test, preds):
  """
  Calculating the RMSE
  """
  return np.sqrt(mean_squared_error(test,preds))

In [ ]:
def MAE(test, preds):
  """
  Calculating the MAE (mean absolute error)
  """
  return(mean_absolute_error(test,preds))

In [ ]:
# Function to map Sumatra
def scatter_map(df, color, title, size, point_title, data, map_style = "satellite", height = 1000, width = 1000):

  """
  This function creates a scatter map given a dataframe with "lat" and "lon" columns.
  If the value to group by is qualitative, but saved as an int, it is recommended to convert the column data to strings before using the map.
  .astype(str) can be used to change the data

  - df: the dataframe with the column information to plot
  - color: column name to be used when coloring or "grouping" the data points (it can be quantitative or qualitative)
  Example: Color by deforestacion fraction or cluster id
  - title: Name for the map
  - size: Name of the column with the size for each point
  - point_title: Name to appear when hovering over a point
  - data: Information to display when hovering over the points (can be a list)
  - map_style: Type of map to plot on
  - height and weight: size of the map
  """

  # Figure for the map
  fig = px.scatter_map(
    df, # Dataframe to extract information from

    # Coordinates
    lat = "lat", lon ="lon",

    hover_name = point_title, # Name for the points
    hover_data = data, # Data to display while hovering
    size = size, # Size for each point
    color = color, # Column to be used to classify and color the data

    # Colors when using discrete classification
    color_discrete_sequence = px.colors.qualitative.Light24,
    zoom = 5, # How zoomed in the map will be generated
    map_style = map_style, # Type of map
    title = title, # Map title

    # Map size
    width = width, height = height)

  fig.show()

In [ ]:
# Function that loads and prepares geojson data for the choropleth map
def prep_geojson():

  """
  First function out of three to create a choropleth map of Sumatra.
  This function loads the geojson map and parses through it to obtain the coordinate information we need to plot.
  It does not need any arguments.
  Returns the original geojson, a dataframe with the geojson information of the provinces,
  and a list with the province names as found in the geojson.
  """

  # Get Indonesia geojson from github
  geojson = requests.get("https://raw.githubusercontent.com/superpikar/indonesia-geojson/master/indonesia-province-simple.json").json()

  # Parsing through the features column of the geojson
  # Features is a dictionary that contains all of the geographical information needed
  geojson_pd = pd.json_normalize(geojson["features"])

  # List to of the provinces of Sumatra as written in the geojson file
  sumatra_prov = ["DI. ACEH", "BENGKULU", "JAMBI", "LAMPUNG", "RIAU", "SUMATERA BARAT", "SUMATERA SELATAN", "SUMATERA UTARA"]

  # Only save rows with Sumatra provinces by checking if they are listed in sumatra_prov
  # properties.Propinsi contains all of the province names in Indonesia
  geojson_pdp = geojson_pd[geojson_pd["properties.Propinsi"].isin(sumatra_prov)]

  # Reset index and drop extra index column
  geojson_pdp = geojson_pdp["properties.Propinsi"].reset_index()
  geojson_pdp = geojson_pdp.drop(columns = "index")

  return geojson, geojson_pdp, sumatra_prov

# Not finished functions for choropleth (almost done)

In [ ]:
# Function to add the geojson information to our data
def prep_choropleth_data(df, geojson_pdp, sumatra_prov):

  """
  Given the dataframe with the information to be plotted and the processed geojson,
  add the geojson labels to the correct rows according to the province.
  df: dataframe with the information we want to plot
  geojson_pdp: dataframe with the map properties extracted
  sumatra_prov: list of the provinces in Sumatra with official geojson name
  """

  # Create function to use with apply









In [ ]:
# Function to make a choropleth map specifically of Sumatra
def choropleth_map(og_geojson, df, ):

  """

  """

  # Figure for choropleth
  fig = go.Figure(
      data=go.Choropleth(geojson=geojson,
      locations=deforest_hex["geojson"], # Name of the provinces
      featureidkey="properties.Propinsi", # Dictionary with region information
      z=deforest_hex["defor_frac"],  # Data to be color-coded
      colorscale="Reds",
      colorbar_title="Deforestation Fraction")
      )


  fig.update_geos(fitbounds="locations", visible=True)
  fig.update_layout(title_text = "Fraction of Deforestation in Sumatra from 2000-2018")

  fig.show()
